# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL compliant with the [MLCommons Croissant](https://mlcommons.org/standards/croissant/) standard.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '')}\n\n{getattr(metadata, 'description', '')}")
print("\nRecord Sets available in dataset (@id):")
for rs in getattr(metadata, 'recordSet', []):
    rs_id = getattr(rs, '@id', str(rs)) if hasattr(rs, '@id') else str(rs)
    print(f"- {rs_id}")

## 2. Data Overview
List available record sets, fields, and their `@id` values from the dataset. 
Use these `@id` fields for all further data referencing steps.

In [ ]:
# Explore and display available record sets and their fields (@id values).
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found (recordSet is empty in metadata). This dataset might define RecordSets via references or files.")
    # Optionally, try listing via dataset.list_record_sets():
    try:
        record_set_ids = dataset.list_record_sets()
        if not record_set_ids:
            print("No record sets found via mlcroissant. Please check the dataset schema.")
        else:
            print("Record sets detected by mlcroissant:")
            for rs_id in record_set_ids:
                print(f"- {rs_id}")
    except Exception as e:
        print(f"Could not list record sets: {e}")
else:
    # List fields/columns for each record set
    for rs in record_sets:
        rs_id = getattr(rs, '@id', str(rs))
        print(f"\nRecord Set: {rs_id}")
        # Fields/columns
        fields = getattr(rs, 'field', []) + getattr(rs, 'column', [])
        if not fields:
            print("  No fields/columns found for this record set.")
        else:
            for fld in fields:
                fld_id = getattr(fld, '@id', str(fld))
                print(f"  - Field/Column: {fld_id}")

## 3. Data Extraction
Load the data from each record set into pandas DataFrames for analysis using `mlcroissant`. All record set and field references must use their `@id` values.

In [ ]:
# Identify usable record sets via mlcroissant
try:
    record_set_ids = dataset.list_record_sets()
    print(f"Available record set @id's: {record_set_ids}")
except Exception as e:
    record_set_ids = []
    print(f"Error when listing record sets: {e}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set '{record_set_id}'. Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading records from '{record_set_id}': {e}")

# Display the head of the first available record set DataFrame
displayed = False
for rsid, df in dataframes.items():
    print(f"\nFirst 5 rows for record set '{rsid}':")
    print(df.head())
    displayed = True
    break
if not displayed:
    print("No data frames loaded from any record set. Cannot continue EDA without data.")

## 4. Exploratory Data Analysis (EDA)
Example: Filter and process a numeric field within a record set.

- Select a field with numeric values for filtering and normalization
- Demonstrate data filtering, normalization, and grouping by another field
- Use field `@id` values for references

In [ ]:
# --- Select demo record set and fields by @id ---
if not dataframes:
    print("No available dataframes from previous steps. Cannot perform EDA.")
else:
    # Get the first available record set for demonstration
    selected_record_set = list(dataframes.keys())[0]
    df = dataframes[selected_record_set]

    print(f"EDA on record set: {selected_record_set}")

    # Try to automatically select a numeric column
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_fields:
        print('No numeric fields available for this record set.')
    else:
        # Take the first numeric field for example
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        # Example threshold for filtering; adjust as needed
        threshold = df[numeric_field].quantile(0.90) if df[numeric_field].nunique()>10 else df[numeric_field].max()/2
        filtered_df = df[df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing up to 5 rows):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}' (mean/std) [top 5 rows]:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt groupby on a likely grouping column (e.g., by 'ward', 'gender', or similar if exists)
        selected_group_field = None
        poss_group_fields = [col for col in df.columns if col.lower() in ['ward','gender','location','county','cluster','group','category']]
        if poss_group_fields:
            selected_group_field = poss_group_fields[0]
        elif len(df.select_dtypes(include=['object','category']).columns) > 0:
            selected_group_field = df.select_dtypes(include=['object','category']).columns[0]

        if selected_group_field and selected_group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(selected_group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped filtered data by '{selected_group_field}' (showing top 5 rows):")
            print(grouped_df.head())
        else:
            print("\nNo suitable string/categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or field relationships from the dataset.

- Plot numeric field histogram
- If grouping field present, plot mean of numeric by group

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No DataFrames to visualize.")
else:
    # Use previous selections (from EDA, kept in variables)
    # Fallback if not found
    try:
        df = dataframes[selected_record_set]
        field_to_plot = numeric_field if 'numeric_field' in locals() else None
        group_field = selected_group_field if 'selected_group_field' in locals() else None
    except Exception:
        field_to_plot = None
        group_field = None

    if field_to_plot and field_to_plot in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[field_to_plot], kde=True)
        plt.title(f"Distribution of {field_to_plot}")
        plt.xlabel(field_to_plot)
        plt.show()
    else:
        print("No valid numeric field to plot.")

    # If grouping possible, plot mean of numeric field by group
    if group_field and group_field in df.columns and field_to_plot:
        # Show top 10 groups
        group_means = df.groupby(group_field)[field_to_plot].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(7,4))
        sns.barplot(y=group_means.index, x=group_means.values, orient='h')
        plt.title(f"Mean {field_to_plot} by {group_field} (Top 10)")
        plt.xlabel(f"Mean {field_to_plot}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load a Croissant-structured dataset using `mlcroissant`
- Explore record sets, fields, and record metadata
- Extract and filter records from each record set using `@id` values
- Perform normalization, grouping, and visualization operations for exploratory data analysis

Review the dataset schema documentation for detailed attribute and field explanations. Refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/usage/) for advanced usage.